# Gift-Eval Head Ablation Analysis

This notebook analyzes the impact of **attention head ablation** on Time Series Foundation Model (TSFM) performance across the Gift-Eval benchmark datasets.

## Overview

**Purpose:** Evaluate model robustness by progressively ablating (zeroing out) attention heads and measuring the effect on forecasting metrics.

**Supported Models:** TimesFM 2.5, Chronos Bolt, Toto

## Analysis Pipeline

1. **Data Loading** — Load ablation results across multiple random seeds and select the best-performing seed per ablation level
2. **Normalization** — Optionally normalize metrics by seasonal naive baseline for fair comparison
3. **Statistical Analysis** — Compute geometric means and standard errors across datasets for each ablation level
4. **Visualization:**
   - **Line plots** — Performance degradation curves as heads are ablated per layer
   - **Bar plots** — Dataset-level comparison of baseline vs. best ablation performance  
   - **Heatmaps** — Per-dataset performance across all ablation levels
5. **Dataset Categorization** — Identify resilient, improved, and vulnerable datasets based on ablation response

## Key Metrics
- Primary: MASE (Mean Absolute Scaled Error) at 0.5 quantile
- Aggregation: Geometric mean across datasets

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean
import pprint

#### Configuration

In [ ]:
model_name = "Toto"

#### Presets

In [ ]:
model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
}

# Select the metric to analyze
SELECTED_METRIC = "eval_metrics/MASE[0.5]"
# SELECTED_METRIC = "eval_metrics/sMAPE[0.5]"

metric_pretty_name = "".join(c for c in SELECTED_METRIC.split("/")[-1] if c.isalpha())

print(f"Analyzing metric: {SELECTED_METRIC}")
print(f"Metric name: {metric_pretty_name}")


In [ ]:
apply_custom_style("../../config/plotting.yaml")
HOME_DIR = os.getenv("HOME")
root_dir = os.path.join(HOME_DIR, "tsfm-lens")  # type: ignore

save_figs = True
figs_save_dir = os.path.join(root_dir, "figs", f"gift-eval_{metric_pretty_name}", model_to_dir_mapping[model_name])
if not os.path.exists(figs_save_dir):
    os.makedirs(figs_save_dir)
if save_figs:
    print(f"Saving figures to: {figs_save_dir}")

## Load Data


In [ ]:
# Model configurations: (layers, num_heads_levels, n_heads_per_layer, layer_descriptions, srank_layers, reverse_srank_layers)
MODEL_CONFIGS = {
    "TimesFM 2.5": {
        "layers": [0, 5, 10, 19],
        "head_levels": [1, 2, 4, 6, 8, 10, 12],
        "n_heads": 16,
        "descriptions": {0: "srank", 5: "reverse srank", 10: "reverse srank", 19: "best random"},
        "srank": [0],
        "reverse_srank": [5, 10],
    },
    "Chronos Bolt": {
        "layers": [0, 2, 5, 11],
        "head_levels": [1, 2, 4, 6, 8, 10],
        "n_heads": 12,
        "descriptions": {0: "best random", 2: "reverse srank", 5: "best random", 11: "best random"},
        "srank": [0],
        "reverse_srank": [2],
    },
    "Toto": {
        "layers": [0, 2, 9, 11],
        "head_levels": [1, 2, 4, 6, 8, 10],
        "n_heads": 12,
        "descriptions": {0: "srank", 2: "best random", 9: "reverse srank", 11: "srank"},
        "srank": [0, 11],
        "reverse_srank": [9],
    },
}

if model_name not in MODEL_CONFIGS:
    raise ValueError(f"Model {model_name} not supported yet")

cfg = MODEL_CONFIGS[model_name]
selected_layers_lst, levels_num_heads_ablated, n_divisor = cfg["layers"], cfg["head_levels"], cfg["n_heads"]
layers_srank, layers_srank_reverse = cfg["srank"], cfg["reverse_srank"]
extra_description_str_by_key = {f"Layer {k}": v for k, v in cfg["descriptions"].items()}

# Build ablation files dictionary for all selected layers
ablation_meaning_str = "Ablation of Heads Across Layers"
ablation_level_meaning_str = "Pct of Heads Ablated per Layer"
ablation_level_meaning_alternative_str = "Number of Heads Ablated per Layer"

ablated_files_dict, ablation_meaning_str_dict = {}, {}
ablation_level_meaning_str_dict, ablation_level_meaning_alternative_str_dict = {}, {}

for layer in selected_layers_lst:
    key = f"Layer {layer}" if not isinstance(layer, list) else f"Layers {'-'.join(map(str, layer))}"
    ablation_meaning_str_dict[key] = f"Ablation of Heads in Layer {layer}"
    ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated"
    ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated"
    ablated_files_dict[key] = {
        0: "original_all_results.csv",
        **{n: f"head_layers_{layer}_heads-{n}_all_results.csv" for n in levels_num_heads_ablated},
        n_divisor: f"head_layers_{layer}_heads-all_all_results.csv",
    }

In [ ]:
pprint.pp(ablated_files_dict)

In [ ]:
random_rseeds_lst = [
    123,
    99,
    42,
    688,
    22,
    33,
    56,
    78,
    357,
    234,
    567,
    890,
    1356,
    2233,
    3566,
    4891,
    222,
    333,
    444,
    555,
    777,
]
srank_rseeds_lst = srank_reverse_rseeds_lst = [99, 42]

# Build metrics directories for each selection strategy
base_dir = os.path.join(root_dir, "results", model_to_dir_mapping[model_name])
make_dirs = lambda seeds, prefix="": {s: os.path.join(base_dir, f"{prefix}rseed-{s}") for s in seeds}

random_metrics_dir_dict = make_dirs(random_rseeds_lst)
srank_metrics_dir_dict = make_dirs(srank_rseeds_lst, "srank_")
srank_reverse_metrics_dir_dict = make_dirs(srank_reverse_rseeds_lst, "srank_reverse_")

for name, d in [
    ("Random", random_metrics_dir_dict),
    ("Stable rank (low to high)", srank_metrics_dir_dict),
    ("Stable rank (high to low)", srank_reverse_metrics_dir_dict),
]:
    print(f"{name} head selection metrics directory: {d}")


In [ ]:
ablated_files_dict.keys()

In [ ]:
# Map layer keys to their metrics directory
layers_srank_set = {f"Layer {l}" for l in layers_srank}
layers_srank_reverse_set = {f"Layer {l}" for l in layers_srank_reverse}
print(f"Layers srank: {layers_srank_set}\nLayers srank reverse: {layers_srank_reverse_set}")


def get_metrics_dir(key):
    if key in layers_srank_set:
        return srank_metrics_dir_dict
    if key in layers_srank_reverse_set:
        return srank_reverse_metrics_dir_dict
    return random_metrics_dir_dict


# Load CSVs, compute gmean per rseed, select best rseed per level
gmean_key = f"gmean_{SELECTED_METRIC}"
best_filepaths_by_key, data_by_key = {}, {}

for key, ablation_files in ablated_files_dict.items():
    print(f"Processing key: {key}")
    best_filepaths_by_key[key], curr_data = {}, {}
    metrics_dir_dict = get_metrics_dir(key)

    for level, filepath in ablation_files.items():
        # Load valid data for each rseed
        rseed_data = {}
        for rseed, metrics_dir in metrics_dir_dict.items():
            df_path = os.path.join(metrics_dir, filepath)
            if not os.path.exists(df_path):
                # print(f"Level {level}, rseed {rseed}: Skipping (not found)")
                continue
            df = pd.read_csv(df_path)
            if len(df) != 97:
                # print(f"Level {level}, rseed {rseed}: Skipping (incomplete)")
                continue
            print(f"Level {level}, rseed {rseed}: Loading")
            df["ablation_level"] = level
            rseed_data[rseed] = (df, df_path)

        if not rseed_data:
            # print(f"Level {level}: No data found, skipping")
            continue

        # Select rseed with lowest geometric mean
        rseed_gmeans = {r: gmean(df[SELECTED_METRIC]) for r, (df, _) in rseed_data.items()}
        best_rseed = min(rseed_gmeans, key=lambda r: rseed_gmeans[r])
        best_df, best_path = rseed_data[best_rseed]

        best_filepaths_by_key[key][level] = {
            "best_rseed": best_rseed,
            "best_filepath": best_path,
            "available_rseeds": list(rseed_data.keys()),
            gmean_key: rseed_gmeans[best_rseed],
            "num_rseeds_evaluated": len(rseed_data),
        }
        curr_data[level] = best_df
        print(
            f"Level {level}: Best rseed={best_rseed} (gmean={rseed_gmeans[best_rseed]:.6f}) from {len(rseed_data)} rseeds"
        )

    data_by_key[key] = pd.concat(curr_data.values(), ignore_index=True)
    print(f"\nTotal: {len(data_by_key[key])} rows, {data_by_key[key]['dataset'].nunique()} unique datasets")

In [ ]:
# Show example of stored filepath information
example_key = list(best_filepaths_by_key.keys())[1]
print(f"Example: {example_key}\n" + "=" * 80)
for level, info in best_filepaths_by_key[example_key].items():
    print(
        f"  Num Heads Ablated {level}: rseed={info['best_rseed']}, gmean={info[gmean_key]:.6f}, "
        f"n={info['num_rseeds_evaluated']} {info['available_rseeds']}"
    )


## Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True
is_normalized = False

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


In [ ]:
data_by_key.keys()

In [ ]:
if NORMALIZE_BY_SEASONAL_NAIVE and not is_normalized:
    seasonal_naive_df = pd.read_csv(os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv"))
    print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets. Normalizing...")
    for key in data_by_key:
        data_by_key[key] = normalize_by_seasonal_naive(data_by_key[key], seasonal_naive_df)
    is_normalized = True
else:
    print("Skipping normalization.")


### Display Available Metrics


In [ ]:
# Display available metrics
metric_columns = [col for col in df.columns if "eval_metrics" in col]
print("Available metrics:")
for i, metric in enumerate(metric_columns, 1):
    print(f"{i}. {metric}")


### Line Plot


In [ ]:
# Prepare data for box plot
box_data_by_key = {}
for key in data_by_key.keys():
    box_data = data_by_key[key][["dataset", "ablation_level", SELECTED_METRIC]].copy()
    # Remove any infinite or NaN values
    box_data = box_data.replace([np.inf, -np.inf], np.nan).dropna()
    box_data_by_key[key] = box_data


In [ ]:
# Calculate median and percentiles for each ablation level and key
stats_by_level_by_key = {}
for key in data_by_key.keys():
    box_data = box_data_by_key[key]
    stats_by_level = (
        box_data.groupby("ablation_level")[SELECTED_METRIC]
        .agg(
            [
                "median",
                "mean",
                lambda x: gmean(x),
                lambda x: np.exp(np.std(np.log(x)) / np.sqrt(len(x))),  # geometric standard error (scale factor)
                lambda x: x.quantile(0.25),
                lambda x: x.quantile(0.75),
            ]
        )
        .rename(columns={"<lambda_0>": "geom_mean", "<lambda_1>": "geom_ste", "<lambda_2>": "q25", "<lambda_3>": "q75"})
        .reset_index()
    )
    stats_by_level_by_key[key] = stats_by_level

In [ ]:
single_layer_colors = plt.cm.get_cmap("cividis")(np.linspace(0.1, 0.95, len(data_by_key.keys())))
colors = list(single_layer_colors)

In [ ]:
extra_description_str_by_key.keys()

In [ ]:
# Create line plot with error bands
fig, ax = plt.subplots(figsize=(4, 5))


# Helper for baseline reference lines
def add_ref_line(y, label, color, va, y_mult):
    ax.axhline(y=y, color=color, linestyle="--", linewidth=2, alpha=0.7)
    ax.text(0, y * y_mult, label, color=color, fontsize=8, fontweight="bold", va=va, ha="left", zorder=100)


for i, key in enumerate(data_by_key):
    stats = stats_by_level_by_key[key]
    x_pct = stats["ablation_level"] / n_divisor * 100

    if i == 0:  # Add baseline references on first iteration
        baseline = stats.loc[stats["ablation_level"] == 0, "geom_mean"].values[0]
        add_ref_line(baseline, f"Baseline ({baseline:.3f})", "red", "top", 0.994)
        add_ref_line(1.01 * baseline, "Baseline + 1%", "orangered", "bottom", 1.004)

    ax.plot(
        x_pct,
        stats["geom_mean"],
        marker="^",
        markerfacecolor="None",
        markeredgecolor=colors[i],
        markeredgewidth=2,
        linewidth=2.5,
        markersize=7,
        color=colors[i],
        alpha=0.8,
        label=f"{key} ({extra_description_str_by_key[key]})",
    )
    ax.fill_between(
        x_pct,
        stats["geom_mean"] * stats["geom_ste"] ** 0.33,
        stats["geom_mean"] / stats["geom_ste"] ** 0.33,
        alpha=0.05,
        color=colors[i],
        zorder=-1,
    )

# Formatting
ax.set(
    xlabel=f"{ablation_level_meaning_str} (%)",
    ylabel=f"{metric_pretty_name} (Geom Mean)",
    ylim=(0.98 * baseline, 1.06 * baseline),
    title=model_name,
)
for lbl in [ax.xaxis.label, ax.yaxis.label, ax.title]:
    lbl.set_fontweight("bold")
ax.grid(True, alpha=0.2)
ax.legend(loc="upper left", frameon=True)

# Set x-axis ticks
all_levels = {lvl for s in stats_by_level_by_key.values() for lvl in s["ablation_level"]}
xticks = sorted(x / n_divisor * 100 for x in all_levels)
ax.set_xticks(xticks)
ax.set_xticklabels([f"{int(x)}" if x == int(x) else f"{x:.1f}" for x in xticks])

plt.tight_layout()
if save_figs:
    save_path = os.path.join(figs_save_dir, f"{ablation_meaning_str}_line_plot_{model_name}.pdf")
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved: {save_path}")
plt.show()

In [ ]:
# Calculate and display percentage change from baseline for each key
for key, stats in stats_by_level_by_key.items():
    base_med = stats.loc[stats["ablation_level"] == 0, "median"].values[0]
    base_gm = stats.loc[stats["ablation_level"] == 0, "geom_mean"].values[0]
    print(f"\n% Change from Baseline for {key}:\n" + "=" * 70)
    for _, r in stats.iterrows():
        n, gm = int(r["ablation_level"]), r["geom_mean"]
        print(
            f"{n} heads ({n / n_divisor * 100:.1f}%): Median {(r['median'] / base_med - 1) * 100:+.2f}%, "
            f"Geom Mean {(gm / base_gm - 1) * 100:+.2f}% (curr: {gm:.3f})"
        )


## Identify Datasets close to Baseline

In [ ]:
data_by_key.keys()

In [ ]:
selected_layer = "Layer 0"
selected_layer_data = data_by_key[selected_layer]
print(f"Selected layer: {selected_layer}")

In [ ]:
# Get baseline and compute relative performance
baseline = selected_layer_data[selected_layer_data["ablation_level"] == 0][["dataset", SELECTED_METRIC]]
baseline.columns = ["dataset", "baseline_value"]
merged_data = selected_layer_data.merge(baseline, on="dataset")
merged_data["rel"] = merged_data[SELECTED_METRIC] / merged_data["baseline_value"]

# max_lvl = merged_data["ablation_level"].max()
max_lvl = 10
max_data = merged_data[merged_data["ablation_level"] == max_lvl]
print(f"Max ablation level: {max_lvl}")


def build_info(df):
    return {
        r["dataset"]: {
            "baseline": r["baseline_value"],
            "max_abl": r[SELECTED_METRIC],
            "rel": r["rel"],
            "all_data": merged_data[merged_data["dataset"] == r["dataset"]].sort_values("ablation_level"),
        }
        for _, r in df.iterrows()
    }


def print_results(title, info, reverse=False):
    print(f"\n{title}:\n" + "=" * 90)
    for i, (ds, d) in enumerate(sorted(info.items(), key=lambda x: x[1]["rel"], reverse=reverse)[:10], 1):
        print(f"{i:2d}. {ds:50s} | Rel: {d['rel']:.4f} | Base: {d['baseline']:.4f} | MaxAbl: {d['max_abl']:.4f}")


# Process dataset categories
categories = [
    ("resilient", max_data[(max_data["rel"] >= 0.95) & (max_data["rel"] <= 1.05)], False),
    ("improved", max_data[max_data["rel"] <= 1.0], False),
    ("vulnerable", max_data[max_data["rel"] >= 1.0], True),
]
datasets_by_cat = {}
for name, df, rev in categories:
    datasets_by_cat[name] = build_info(df)
    print(f"Found {len(datasets_by_cat[name])} {name} datasets")
    print_results(f"{name.title()} Datasets", datasets_by_cat[name], rev)

datasets_resilient, datasets_improved, datasets_vulnerable = [
    datasets_by_cat[k] for k in ["resilient", "improved", "vulnerable"]
]


In [ ]:
# Plot improved datasets with their highest ablation level that still shows improvement
MAX_DATASETS = 12

# Pivot data: datasets × ablation levels
pivot = merged_data.pivot(index="dataset", columns="ablation_level", values=SELECTED_METRIC)
ablation_cols = [c for c in pivot.columns if c != 0]

# For each dataset, find highest ablation level with improvement (value < baseline)
plot_data = []
for ds in pivot.index:
    base = pivot.loc[ds, 0]
    improving = [c for c in ablation_cols if pivot.loc[ds, c] < base]
    if not improving:
        continue
    best_lvl = max(improving)
    best_val = pivot.loc[ds, best_lvl]
    plot_data.append(
        {"dataset": ds, "baseline": base, "abl_val": best_val, "abl_lvl": best_lvl, "impr": base - best_val}
    )

plot_data = sorted(plot_data, key=lambda x: x["impr"], reverse=True)[:MAX_DATASETS]
print(f"Datasets with improvement: {len(plot_data)}/{len(pivot)} | Displaying top {len(plot_data)}")

if not plot_data:
    print("No improving datasets found")
else:
    fig, ax = plt.subplots(figsize=(6, max(6, len(plot_data) * 0.5)))
    y_pos = np.arange(len(plot_data))[::-1]
    bw = 0.4

    ax.barh(y_pos - bw / 2, [d["baseline"] for d in plot_data], bw, label="Original", color="tab:green", alpha=0.6)
    bars = ax.barh(
        y_pos + bw / 2,
        [d["abl_val"] for d in plot_data],
        bw,
        label="Max Improving Ablation",
        color="tab:blue",
        alpha=0.6,
    )

    for bar, d in zip(bars, plot_data):
        ax.text(
            bar.get_width(),
            bar.get_y() + bar.get_height() / 2,
            f" ({d['abl_lvl']} heads)",
            ha="left",
            va="center",
            fontsize=7,
            fontweight="bold",
        )

    ax.set(yticks=y_pos, ylim=(y_pos.min() - 0.5, y_pos.max() + 1.5))
    ax.set_yticklabels([d["dataset"] for d in plot_data], fontsize=8)
    ax.set_xlabel(f"$\\mathbf{{{metric_pretty_name}}}$", fontsize=12)
    ax.set_ylabel("Dataset", fontsize=12, fontweight="bold")
    ax.set_title(f"{ablation_meaning_str} ({model_name})", fontsize=13, fontweight="bold", pad=10)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=10)
    ax.grid(True, alpha=0.3, axis="x")

    plt.tight_layout()
    if save_figs:
        save_path = os.path.join(
            figs_save_dir, f"{ablation_meaning_str}_datasets_improvement_bar_plot_{model_name}.pdf"
        )
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    print("✓ Blue bars = highest ablation level still showing improvement, sorted by improvement")


In [ ]:
import seaborn as sns
from matplotlib.colors import Normalize

# Pivot and sort datasets by name/freq/term
heatmap_data = box_data.pivot(index="dataset", columns="ablation_level", values=SELECTED_METRIC)


def sort_key(name):
    parts = name.split("/")
    if len(parts) != 3:
        return (name, 0, 0)
    ds, freq, term = parts
    freq_ord = {"H": 1, "D": 2, "W": 3, "M": 4, "Q": 5, "Y": 6}
    freq_val = int(freq[:-1]) if freq.endswith("T") and freq[:-1].isdigit() else freq_ord.get(freq, 10) * 1000
    return (ds, freq_val, {"short": 0, "medium": 1, "long": 2}.get(term, 3))


heatmap_data = heatmap_data.reindex(sorted(heatmap_data.index, key=sort_key))
if len(heatmap_data) > 100:
    print(f"Showing top 50/{len(heatmap_data)} datasets")
    heatmap_data = heatmap_data.head(50)

# Split into 3 parts and plot
n, vmin, vmax = len(heatmap_data), heatmap_data.min().min(), 3
splits = [heatmap_data.iloc[i * n // 3 : (i + 1) * n // 3] for i in range(3)]
fig, axes = plt.subplots(1, 3, figsize=(20, max(8, n // 3 * 0.3)))

for i, (ax, data) in enumerate(zip(axes, splits)):
    sns.heatmap(data, annot=True, cmap="YlOrRd", vmin=vmin, vmax=vmax, cbar=False, linewidths=0.5, ax=ax)
    ax.set(xlabel=ablation_level_meaning_alternative_str, ylabel="Dataset" if i == 0 else "")
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontweight("bold")
    ax.set_title(f"Datasets {i * n // 3 + 1}-{i * n // 3 + len(data)}", fontsize=12, fontweight="bold")

# Shared colorbar and title
fig.subplots_adjust(right=0.92)

sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=fig.add_axes((0.94, 0.15, 0.015, 0.7)))
cbar.set_label(f"$\\mathbf{{{metric_pretty_name}}}$", fontsize=12, fontweight="bold")
fig.suptitle(f"{ablation_meaning_str} ({model_name})", fontsize=20, fontweight="bold", y=0.96)

plt.tight_layout(rect=(0, 0, 0.93, 0.96))
if save_figs:
    save_path = os.path.join(figs_save_dir, f"{ablation_meaning_str}_datasets_heatmap_{model_name}.pdf")
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved: {save_path}")
plt.show()
print(f"Showing {n} datasets split into 3 columns")
